# 02 — Estructura y calidad de los datos

TFM — Predicción y segmentación de las emisiones de CO₂ de los vehículos
nuevos matriculados en la UE.

Fuente: Agencia Europea de Medio Ambiente (EEA), tabla `co2cars_2024Fv30`.
Muestra pseudoaleatoria reproducible del 10 % (1.076.533 vehículos).

Objetivo: describir la estructura del conjunto, cuantificar los valores
ausentes por variable e identificar los problemas de calidad que condicionan
el preprocesado. Alimenta los apartados 3.1 y 4.2 de la memoria.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

RAIZ = Path.cwd().parent
PARQUET = RAIZ / "datos" / "crudo" / "co2cars_2024_muestra1de10.parquet"
TABLA = f"'{PARQUET.as_posix()}'"

# DuckDB consulta el Parquet directamente, sin cargarlo antes en memoria
con = duckdb.connect()
total = con.execute(f"SELECT COUNT(*) FROM {TABLA}").fetchone()[0]

print("Fichero:", PARQUET.name)
print(f"Registros: {total:,}".replace(",", "."))

Fichero: co2cars_2024_muestra1de10.parquet
Registros: 1.076.533


## 1. Esquema del conjunto

Tipo de dato que DuckDB deduce de cada columna del fichero Parquet.

In [2]:
esquema = con.execute(f"DESCRIBE SELECT * FROM {TABLA}").df()
print(f"Número de variables: {len(esquema)}\n")
esquema[["column_name", "column_type"]]

Número de variables: 28



,column_name,column_type
0,ID,BIGINT
1,MS,VARCHAR
2,Mp,VARCHAR
3,Mh,VARCHAR
4,Man,VARCHAR
5,Mk,VARCHAR
6,Cn,VARCHAR
7,Ct,VARCHAR
8,Cr,VARCHAR
9,M (kg),DOUBLE


## 2. Valores ausentes por variable

La consulta se construye dinámicamente sobre la lista de columnas.

In [3]:
columnas = esquema["column_name"].tolist()
expresiones = ",\n    ".join(
    f'SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END) AS "{c}"'
    for c in columnas
)
nulos = con.execute(f"SELECT\n    {expresiones}\nFROM {TABLA}").df().T

nulos.columns = ["nulos"]
nulos["pct_nulos"] = (nulos["nulos"] / total * 100).round(2)
nulos.sort_values("nulos", ascending=False)

,nulos,pct_nulos
At2 (mm),1076533.0,100.00
At1 (mm),1076533.0,100.00
W (mm),1076533.0,100.00
Enedc (g/km),1076533.0,100.00
Z (Wh/km),845986.0,78.58
Erwltp (g/km),485923.0,45.14
Fc,167045.0,15.52
Ec (cm3),156388.0,14.53
Mt,15015.0,1.39
Ewltp (g/km),1485.0,0.14


## 3. Cardinalidad de las variables categóricas

El número de valores distintos decide qué variables admiten codificación
one-hot y cuáles habría que agrupar antes de usarlas.

In [4]:
categoricas = esquema.loc[esquema["column_type"] == "VARCHAR",
                          "column_name"].tolist()

resumen_cat = [
    {"variable": c,
     "valores_distintos": con.execute(
         f'SELECT COUNT(DISTINCT "{c}") FROM {TABLA}').fetchone()[0]}
    for c in categoricas
]
pd.DataFrame(resumen_cat).sort_values("valores_distintos", ascending=False)

,variable,valores_distintos
5,Cn,3735
12,Dr,366
4,Mk,243
2,Mh,105
3,Man,103
10,IT,96
11,Ech,89
0,MS,29
1,Mp,12
8,Ft,10


## 4. Valores de las categóricas de baja cardinalidad

In [5]:
for c in ["Mp", "Ft", "Fm", "Ct", "Cr", "Status"]:
    print(f"\n--- {c} ---")
    print(con.execute(f'''
        SELECT "{c}" AS valor, COUNT(*) AS n
        FROM {TABLA}
        GROUP BY "{c}"
        ORDER BY n DESC
    ''').df().to_string(index=False))


--- Mp ---
                       valor      n
                  VOLKSWAGEN 283377
                  STELLANTIS 173276
   RENAULT-NISSAN-MITSUBISHI 146233
         SUBARU-MAZDA-TOYOTA 103592
                         BMW  71851
                              68397
            MERCEDES-BENZ AG  58851
  VOLVO CARS POLESTAR SUZUKI  48884
        HYUNDAI MOTOR EUROPE  43839
                         KIA  41720
                        FORD  33163
KG MOBILITY GREAT WALL MOTOR   3350

--- Ft ---
          valor      n
         petrol 644865
         diesel 162307
       electric 156307
petrol/electric  72716
            lpg  32220
diesel/electric   4623
            e85   3124
             ng    302
       hydrogen     64
        unknown      5

--- Fm ---
valor      n
    M 468110
    H 341985
    E 156307
    P  77339
    B  32242
    F    546
           4

--- Ct ---
valor       n
   M1 1062754
  M1G   12867
          861
   N1      42
   N2       8
  N1G       1

--- Cr ---
valor       n
   

## 5. Diccionario de variables

Base de la tabla 1 de la memoria. `SUMMARIZE` recorre el fichero una sola vez
y devuelve tipo, porcentaje de valores ausentes y número aproximado de valores
distintos de cada columna.

In [6]:
resumen = con.execute(f"SUMMARIZE SELECT * FROM {TABLA}").df()

diccionario = resumen[["column_name", "column_type", "count",
                       "null_percentage", "approx_unique"]]
pd.set_option("display.width", 200)
print(diccionario.to_string(index=False))

  column_name column_type   count  null_percentage  approx_unique
           ID      BIGINT 1076533             0.00        1251278
           MS     VARCHAR 1076533             0.00             32
           Mp     VARCHAR 1076533             0.00             13
           Mh     VARCHAR 1076533             0.00            110
          Man     VARCHAR 1076533             0.00            103
           Mk     VARCHAR 1076533             0.00            274
           Cn     VARCHAR 1076533             0.00           3190
           Ct     VARCHAR 1076533             0.00              6
           Cr     VARCHAR 1076533             0.00              2
       M (kg)      DOUBLE 1076533             0.00           1716
           Mt      DOUBLE 1076533             1.39           1890
 Ewltp (g/km)      DOUBLE 1076533             0.14            459
 Enedc (g/km)     INTEGER 1076533           100.00              0
       W (mm)     INTEGER 1076533           100.00              0
     At1 (